# Elaborazione video frame-by-frame con OpenCV

**Corso:** Trattamento di Dati Multimediali   
**Strumenti:** Python, OpenCV (`opencv-python`), NumPy, Matplotlib

---

In questa lezione costruiamo una **pipeline di elaborazione video personalizzabile**: leggiamo un file video frame per frame, applichiamo filtri di elaborazione a ciascun frame e salviamo il risultato su un nuovo file video.

Ogni frame di un video è a tutti gli effetti un'immagine. Le tecniche di elaborazione studiate nelle lezioni precedenti (manipolazione pixel, analisi dell'istogramma, conversione degli spazi colore) si applicano quindi direttamente, frame per frame, all'intero flusso video.

## 0. Importazione delle librerie

La costante `CODEC` seleziona il codec usato da `cv2.VideoWriter` in tutto il notebook: `avc1` (H.264) su macOS, `mp4v` su Linux e Google Colab. Il motivo è spiegato nella sezione 4.1.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import time

CODEC = "avc1" if sys.platform == "darwin" else "mp4v"

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version:  {np.__version__}")
print(f"Codec VideoWriter: {CODEC}")


## 1. Struttura di un video digitale

Un video digitale è una sequenza ordinata di **frame** (immagini statiche) riprodotti a una frequenza fissa chiamata **frame rate**, espressa in fotogrammi al secondo (fps).

Le principali proprietà di un video sono:

- **Risoluzione** (larghezza × altezza in pixel): definisce la dimensione spaziale di ogni frame.
- **Frame rate** (fps): definisce la fluidità del movimento. Valori tipici: 24 fps (cinema), 25/30 fps (broadcast), 60 fps (gaming/sport).
- **Durata** (in secondi): ricavabile come `numero_totale_frame / fps`.
- **Codec**: algoritmo usato per comprimere e decomprimere il flusso video (es. H.264/AVC, H.265/HEVC, VP9).

In OpenCV, l'oggetto `cv2.VideoCapture` permette di aprire un file video (o un dispositivo di acquisizione) e accedere ai suoi frame uno alla volta, come se si scorresse una lista di immagini.

## 2. Lettura e ispezione di un file video

### 2.1 Aprire il video e leggere i metadati

`VideoCapture` espone alcune proprietà accessibili tramite il metodo `.get()`, usando le costanti `cv2.CAP_PROP_*`.

**Nota su `CAP_PROP_FRAME_COUNT`**

Il numero di frame restituito da `CAP_PROP_FRAME_COUNT` è una **stima** ricavata dai metadati del contenitore (durata × frame rate, o indici del file), non il risultato di un conteggio. Può quindi differire di qualche unità dal numero di frame effettivamente decodificabili: lo verificheremo nella sezione 4.2 confrontando i frame scritti con quelli dichiarati. Per lo stesso motivo la durata calcolata qui è una durata *stimata*.

**Nota su `cap.release()`**

La chiamata `cap.release()` dice a OpenCV di chiudere il file video e liberare le risorse associate: l'handle al file, i buffer interni e il decoder. Senza di essa il file rimane aperto finché il garbage collector di Python non distrugge l'oggetto, il che può avvenire in momenti imprevedibili.

Nel contesto di un notebook, dove le celle vengono rieseguite frequentemente, omettere `cap.release()` può causare:

- **file bloccato**: su Windows un file video aperto non può essere sovrascritto o spostato da altri processi;
- **accumulo di handle**: ogni riesecuzione della cella apre un nuovo `VideoCapture` sullo stesso file senza chiudere il precedente;
- **output corrotto**: su `VideoWriter`, il trailer del container (che contiene la durata totale e altri metadati) viene scritto solo al momento del rilascio — omettere `out.release()` produce un file video non riproducibile correttamente.

Il pattern corretto in Python sarebbe un context manager (`with cv2.VideoCapture(...) as cap`), ma OpenCV non supporta nativamente il protocollo `with`. La chiamata esplicita a `.release()` è quindi la forma standard.

In [ ]:
VIDEO_PATH = "sample2.mp4"   # sostituire con il percorso del proprio file video

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Impossibile aprire il file: {VIDEO_PATH}")

fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration     = total_frames / fps if fps > 0 else 0

print(f"Risoluzione    : {width} x {height} px")
print(f"Frame rate     : {fps:.2f} fps")
print(f"Frame totali   : {total_frames}")
print(f"Durata stimata : {duration:.2f} s")

cap.release()

### 2.2 Lettura e visualizzazione del primo frame

OpenCV legge i frame in formato **BGR** (Blue-Green-Red). Prima di visualizzare un frame con Matplotlib è necessario invertire l'ordine dei canali con `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)`.

Questo aspetto è già noto dalle lezioni precedenti; lo richiamiamo brevemente perché in questa lezione lavoriamo esclusivamente con OpenCV.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()   # ret=True se la lettura ha avuto successo

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 4.5))
    plt.imshow(frame_rgb)
    plt.title("Frame 0 (primo frame)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Lettura del primo frame fallita.")

cap.release()

### 2.3 Estrazione di frame a intervalli regolari

Una tecnica comune per avere una panoramica del contenuto di un video è estrarre un frame ogni `N` secondi e visualizzarli in griglia.

Il tempo associato a ciascun frame si calcola dal suo indice: `t = indice_frame / fps`. Usare l'indice, anziché moltiplicare il numero progressivo della miniatura per l'intervallo, evita che le etichette slittino quando il frame rate non è intero (ad es. 29,97 fps).


In [ ]:
INTERVALLO_S = 2   # estrai un frame ogni 2 secondi

cap = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS)
step = max(1, int(fps * INTERVALLO_S))

thumbnails = []
frame_idx  = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % step == 0:
        thumbnails.append((frame_idx / fps, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    frame_idx += 1

cap.release()
print(f"Frame estratti: {len(thumbnails)}")

# Visualizzazione in griglia
n_cols = 4
n_rows = max(1, (len(thumbnails) + n_cols - 1) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = np.array(axes).flatten()

for i, ax in enumerate(axes):
    if i < len(thumbnails):
        t, img = thumbnails[i]
        ax.imshow(img)
        ax.set_title(f"t = {t:.2f} s", fontsize=9)
    ax.axis("off")

plt.suptitle("Thumbnail estratti dal video", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Applicazione di filtri frame-by-frame

In questa sezione definiamo quattro filtri di elaborazione e costruiamo una funzione `apply_filter()` che li rende intercambiabili. Ogni filtro è presentato nella propria cella markdown prima del codice.

### 3.1 Conversione in scala di grigi

La conversione in scala di grigi elimina l'informazione cromatica e conserva un solo valore di intensità per pixel. OpenCV usa la formula pesata ITU-R BT.601, già incontrata nelle lezioni precedenti:

$$Y' = 0.299 \cdot R' + 0.587 \cdot G' + 0.114 \cdot B'$$

I valori dei canali letti da un file video sono codificati con correzione gamma (per questo si indicano con l'apice: $R'$, $G'$, $B'$). La combinazione pesata di valori corretti in gamma si chiama **luma** ($Y'$) e approssima la **luminanza** ($Y$), che invece si calcola su valori lineari di intensità luminosa. È la stessa componente $Y'$ dello spazio YCbCr usato nella codifica video.

Il risultato è un'immagine a **singolo canale**. Per salvarlo in un video (che richiede immagini a 3 canali) è necessario riconvertirlo con `cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)`.


In [ ]:
def filtro_grayscale(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

### 3.2 Sfocatura gaussiana

La sfocatura gaussiana applica una **convoluzione** con un kernel gaussiano 2D a ciascun canale dell'immagine. Ogni pixel del risultato è una media pesata dei pixel vicini, con pesi decrescenti allontanandosi dal centro (distribuzione di Gauss).

Parametri principali:
- `ksize`: dimensione del kernel (deve essere dispari, es. `(15, 15)`). Kernel più grandi producono una sfocatura più marcata.
- `sigmaX`: deviazione standard lungo l'asse X. Se impostato a 0, OpenCV lo calcola automaticamente dalla dimensione del kernel.

La sfocatura gaussiana è spesso usata come passo di pre-processing per ridurre il rumore prima di filtri più sensibili (es. Canny).

In [ ]:
def filtro_blur(frame_bgr, ksize=(15, 15)):
    return cv2.GaussianBlur(frame_bgr, ksize, sigmaX=0)

### 3.3 Rilevamento dei bordi con Canny

L'algoritmo di Canny individua i **bordi** dell'immagine, ossia le zone in cui l'intensità varia bruscamente. Nella formulazione originale le fasi sono:

1. Sfocatura gaussiana (riduzione del rumore).
2. Calcolo del gradiente dell'intensità (operatori di Sobel).
3. Non-maximum suppression: si conservano solo i massimi locali lungo la direzione del gradiente.
4. Double thresholding: i pixel vengono classificati come "forti" (> `threshold2`), "deboli" (`threshold1` < p < `threshold2`) o da scartare.
5. Edge tracking by hysteresis: i pixel deboli connessi a pixel forti vengono accettati come bordi.

**Attenzione:** `cv2.Canny` **non** esegue il passo 1. Calcola il gradiente direttamente sull'immagine ricevuta, quindi la riduzione del rumore va fatta esplicitamente prima (ad es. con `cv2.GaussianBlur`). Su un'immagine rumorosa la differenza è netta: il rumore genera molti falsi bordi. Nella sezione 5 lo osserveremo componendo `blur` e `canny` in una pipeline.

I parametri `threshold1` e `threshold2` controllano la sensibilità del rilevatore. Il risultato è un'immagine binaria a singolo canale.


In [ ]:
def filtro_canny(frame_bgr, threshold1=50, threshold2=150):
    blur = cv2.GaussianBlur(frame_bgr, (15, 15), sigmaX=0)
    gray  = cv2.cvtColor(blur, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, threshold1, threshold2)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

### 3.4 Sogliatura adattiva

Nella sogliatura globale si confronta l'intensità di ogni pixel con una soglia fissa valida per tutta l'immagine. Questo approccio fallisce in presenza di illuminazione non uniforme: zone in ombra o sovraesposte verrebbero classificate erroneamente.

La **sogliatura adattiva** risolve il problema calcolando una soglia locale per ogni pixel. Il procedimento è il seguente:

1. Per ogni pixel si considera un intorno quadrato di dimensione `blockSize × blockSize`.
2. Si calcola la media (o la media gaussiana pesata) delle intensità dei pixel nell'intorno.
3. Si sottrae la costante `C` a tale media: il risultato è la soglia locale per quel pixel.
4. Il pixel viene impostato a **bianco** (255) se la sua intensità è maggiore della soglia locale, a **nero** (0) altrimenti.

Il risultato è un'immagine binaria in cui la decisione per ogni pixel dipende esclusivamente dal suo contesto locale, indipendentemente dall'illuminazione globale.

In [ ]:
def filtro_adaptive_threshold(frame_bgr, blockSize=11, C=2):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(
        gray,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY,
        blockSize=blockSize,
        C=C
    )
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

### 3.5 Funzione dispatcher `apply_filter`

Raccogliamo i quattro filtri in un'unica funzione che accetta un parametro `mode` per selezionare il filtro desiderato.

In [ ]:
FILTRI_DISPONIBILI = ["grayscale", "blur", "canny", "adaptive_threshold"]

def apply_filter(frame_bgr, mode="grayscale"):
    if mode == "grayscale":
        return filtro_grayscale(frame_bgr)
    elif mode == "blur":
        return filtro_blur(frame_bgr)
    elif mode == "canny":
        return filtro_canny(frame_bgr)
    elif mode == "adaptive_threshold":
        return filtro_adaptive_threshold(frame_bgr)
    else:
        raise ValueError(f"Filtro '{mode}' non riconosciuto. Scegliere tra: {FILTRI_DISPONIBILI}")

### 3.6 Preview dei filtri su un frame campione

Prima di processare l'intero video, verifichiamo visivamente l'effetto di ciascun filtro su un frame rappresentativo.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)  # posizionati sul frame centrale
ret, frame_campione = cap.read()
cap.release()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

titoli = ["Originale"] + FILTRI_DISPONIBILI
frames = [cv2.cvtColor(frame_campione, cv2.COLOR_BGR2RGB)] + [
    cv2.cvtColor(apply_filter(frame_campione, m), cv2.COLOR_BGR2RGB)
    for m in FILTRI_DISPONIBILI
]

for ax, img, titolo in zip(axes, frames, titoli):
    ax.imshow(img)
    ax.set_title(titolo, fontsize=10)
    ax.axis("off")

plt.suptitle("Confronto filtri su frame campione", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Pipeline di processing e salvataggio del video

### 4.1 Funzione `process_video`

La funzione legge il video di input frame per frame, applica il filtro scelto e scrive il risultato in un nuovo file video tramite `cv2.VideoWriter`.

`VideoWriter` richiede:
- Il **path** del file di output.
- Un **fourcc** (Four-Character Code) che identifica il codec.
- Il **frame rate** dell'output (tipicamente uguale all'input).
- La **risoluzione** (larghezza, altezza) dei frame di output.

**Nota sul codec**

I codec disponibili dipendono dal sistema operativo e dalla build di OpenCV:

- su **macOS** OpenCV scrive i video tramite AVFoundation e il codec da usare è `avc1` (H.264); con `mp4v` la scrittura può fallire o produrre un file non riproducibile;
- su **Linux e Google Colab** il pacchetto `opencv-python` installato con pip non dispone di un encoder H.264: con `avc1` il `VideoWriter` non si apre, mentre `mp4v` (MPEG-4 Part 2) funziona.

Per questo il notebook usa la costante `CODEC`, definita all'inizio in base al sistema operativo.

Quando il codec richiesto non è disponibile, OpenCV **non solleva eccezioni**: `cv2.VideoWriter(...)` restituisce comunque un oggetto, e le successive chiamate a `out.write()` non scrivono nulla. Per intercettare il problema prima di eseguire l'intero loop si controlla `out.isOpened()` subito dopo la creazione del `VideoWriter`:

```python
if not out.isOpened():
    cap.release()
    raise RuntimeError("VideoWriter non riuscito ad aprire il file di output.")
```

Senza questo controllo il loop verrebbe completato integralmente senza produrre nulla, e l'errore sarebbe scoperto solo alla fine. Il controllo non garantisce però che il file finale sia riproducibile: per questo, nella sezione 4.3, rileggiamo un frame dal video prodotto.


In [ ]:
def process_video(input_path, output_path, mode="grayscale", codec=CODEC, verbose=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    if os.path.exists(output_path):
        os.remove(output_path)

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*codec)
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Verificare il codec (avc1 su macOS, mp4v su Linux/Colab)."
        )

    t_start = time.perf_counter()
    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        filtered = apply_filter(frame, mode)
        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()

    elapsed    = time.perf_counter() - t_start
    throughput = n_frame / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"Filtro         : {mode}")
        print(f"Frame scritti  : {n_frame} / {total}")
        print(f"Tempo totale   : {elapsed:.2f} s")
        print(f"Throughput     : {throughput:.1f} frame/s")
        print(f"Output salvato : {output_path}")

    return n_frame, elapsed

### 4.2 Esecuzione della pipeline

Modifica la variabile `FILTRO_SCELTO` per selezionare il filtro da applicare.

Osserva nell'output il confronto tra *frame scritti* e *frame totali*: se i due numeri differiscono, è l'effetto della stima di `CAP_PROP_FRAME_COUNT` descritta nella sezione 2.1.

Su macOS può comparire l'avviso `WARNING: -finishWriting should not be called on the main thread`, emesso da AVFoundation alla chiusura del file. È innocuo e non indica un errore nella scrittura.


In [ ]:
FILTRO_SCELTO = "adaptive_threshold"   # scegliere tra: grayscale, blur, canny, adaptive_threshold

output_path = f"output_{FILTRO_SCELTO}.mp4"
n_frame, elapsed = process_video(VIDEO_PATH, output_path, mode=FILTRO_SCELTO)

### 4.3 Confronto visivo: frame originale vs. frame processato

Estraiamo il frame centrale dal video di output e lo affianchiamo al corrispondente frame originale.

In [ ]:
def leggi_frame_n(path, n):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, n)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

frame_centrale = n_frame // 2
frame_orig = leggi_frame_n(VIDEO_PATH,  frame_centrale)
frame_proc = leggi_frame_n(output_path, frame_centrale)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame_orig, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Originale — frame {frame_centrale}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(frame_proc, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Filtro: {FILTRO_SCELTO} — frame {frame_centrale}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Pipeline personalizzabile con filtri in cascata

Fino a questo punto ogni video veniva processato con un solo filtro. È possibile comporre più filtri in sequenza: l'output di ogni filtro diventa l'input del successivo.

### 5.1 Definizione della pipeline

In [ ]:
def apply_pipeline(frame_bgr, pipeline):
    """Applica in sequenza una lista di filtri al frame."""
    result = frame_bgr.copy()
    for mode in pipeline:
        result = apply_filter(result, mode)
    return result


def process_video_pipeline(input_path, output_path, pipeline, codec=CODEC, verbose=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    if os.path.exists(output_path):
        os.remove(output_path)

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*codec)
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Verificare il codec (avc1 su macOS, mp4v su Linux/Colab)."
        )

    t_start = time.perf_counter()
    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        filtered = apply_pipeline(frame, pipeline)
        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()

    elapsed    = time.perf_counter() - t_start
    throughput = n_frame / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"Pipeline       : {' -> '.join(pipeline)}")
        print(f"Frame scritti  : {n_frame} / {total}")
        print(f"Tempo totale   : {elapsed:.2f} s")
        print(f"Throughput     : {throughput:.1f} frame/s")
        print(f"Output salvato : {output_path}")

    return n_frame, elapsed

### 5.2 Esecuzione della pipeline personalizzata

Modifica la lista `MIA_PIPELINE` aggiungendo, rimuovendo o riordinando i filtri.

> **Attenzione alla compatibilità tra filtri**: `adaptive_threshold` e `canny` restituiscono immagini binarie. Applicare `blur` prima di essi riduce il rumore prima della sogliatura; applicarlo dopo attenua i contorni. Osserva come cambia il risultato al variare dell'ordine.

In [ ]:
MIA_PIPELINE = [ "blur","canny"]   # personalizza la sequenza di filtri

output_pipeline = "output_pipeline.mp4"
process_video_pipeline(VIDEO_PATH, output_pipeline, pipeline=MIA_PIPELINE)

### 5.3 Confronto visivo degli stadi della pipeline

Visualizziamo l'effetto di ogni stadio della pipeline su un frame campione.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)  # posizionati sul frame centrale
ret, frame_test = cap.read()
cap.release()

stadi     = [frame_test]
etichette = ["Originale"]
corrente  = frame_test.copy()

for filtro in MIA_PIPELINE:
    corrente = apply_filter(corrente, filtro)
    stadi.append(corrente.copy())
    etichette.append(filtro)

fig, axes = plt.subplots(1, len(stadi), figsize=(5 * len(stadi), 4))
for ax, img, label in zip(axes, stadi, etichette):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=10)
    ax.axis("off")

plt.suptitle(f"Stadi della pipeline: {' -> '.join(etichette)}", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Sfida opzionale — Overlay di testo sui frame

`cv2.putText` permette di sovrapporre testo a ogni frame prima della scrittura sul video di output. È utile per aggiungere il numero del frame, il timestamp o etichette diagnostiche.

**Parametri principali di `cv2.putText`:**
- `img`: frame su cui scrivere (modificato in-place).
- `text`: stringa da visualizzare.
- `org`: coordinate (x, y) dell'angolo in basso a sinistra del testo.
- `fontFace`: stile del font (es. `cv2.FONT_HERSHEY_SIMPLEX`).
- `fontScale`: fattore di scala del font, relativo all'altezza base del font Hershey (circa 22 px a scala 1). Per mantenere il testo leggibile a qualunque risoluzione conviene ricavarlo dall'altezza del frame.
- `color`: colore in BGR.
- `thickness`: spessore del tratto in pixel (anch'esso da scalare con la risoluzione).
- `lineType`: `cv2.LINE_AA` per anti-aliasing (testo più pulito).

In [ ]:
def process_video_con_overlay(input_path, output_path, mode="grayscale", codec=CODEC):
    cap    = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if os.path.exists(output_path):
        os.remove(output_path)

    fourcc = cv2.VideoWriter_fourcc(*codec)
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Verificare il codec (avc1 su macOS, mp4v su Linux/Colab)."
        )

    font_scale = height / 1000          # 0.5 a 480p, 1.08 a 1080p
    spessore   = max(1, round(height / 540))

    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        filtered  = apply_filter(frame, mode)
        timestamp = n_frame / fps
        testo     = f"Frame: {n_frame:04d}  |  t = {timestamp:.2f} s  |  filtro: {mode}"

        cv2.putText(
            filtered, testo,
            org=(10, height - int(20 * font_scale)),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=font_scale,
            color=(200, 200, 200),
            thickness=spessore,
            lineType=cv2.LINE_AA
        )

        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()
    print(f"Video con overlay salvato: {output_path} ({n_frame} frame)")
    return n_frame


n_overlay = process_video_con_overlay(VIDEO_PATH, "output_overlay.mp4", mode="grayscale")

# Verifica su un frame campione
frame_ov = leggi_frame_n("output_overlay.mp4", n_overlay // 2)
if frame_ov is not None:
    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(frame_ov, cv2.COLOR_BGR2RGB))
    plt.title("Frame con overlay di testo")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Confronto del throughput tra filtri

Il costo computazionale dei filtri varia. Per confrontarlo misuriamo il throughput (frame/secondo) in due modi diversi:

- **solo filtro**: si cronometra soltanto la chiamata `apply_filter`, cioè il lavoro di elaborazione dell'immagine;
- **catena completa**: si cronometra l'intero ciclo lettura → filtro → scrittura, che include anche la **decodifica** del video di input e la **codifica** del video di output.

La decodifica e la codifica hanno un costo che non dipende dal filtro scelto. Confrontando le due misure si vede quanta parte del tempo totale è effettivamente dovuta al filtro e quanta all'input/output video.

Per contenere i tempi di esecuzione la misura usa i primi `MAX_FRAME` frame del video. I valori ottenuti dipendono dall'hardware, dal sistema operativo, dal codec e dalla risoluzione del video: vanno interpretati come confronto relativo sulla propria macchina, non come valori assoluti.


In [ ]:
MAX_FRAME = 240   # 10 s a 24 fps


def misura_throughput(input_path, mode, codec=CODEC, max_frame=MAX_FRAME):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_tmp = f"_tmp_{mode}.mp4"
    if os.path.exists(out_tmp):   # su macOS AVFoundation non sovrascrive file esistenti
        os.remove(out_tmp)
    out = cv2.VideoWriter(out_tmp, cv2.VideoWriter_fourcc(*codec), fps, (width, height))
    if not out.isOpened():
        cap.release()
        raise RuntimeError(f"VideoWriter non riuscito ad aprire '{out_tmp}'.")

    t_filtro = 0.0
    n = 0
    t_start = time.perf_counter()

    while n < max_frame:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        filtered = apply_filter(frame, mode)
        t_filtro += time.perf_counter() - t0
        out.write(filtered)
        n += 1

    t_totale = time.perf_counter() - t_start
    cap.release()
    out.release()
    if os.path.exists(out_tmp):
        os.remove(out_tmp)

    return n / t_filtro, n / t_totale


risultati = {}
for filtro in FILTRI_DISPONIBILI:
    solo_filtro, catena = misura_throughput(VIDEO_PATH, filtro)
    risultati[filtro] = (solo_filtro, catena)
    print(f"{filtro:<20} solo filtro: {solo_filtro:8.1f} frame/s   catena completa: {catena:6.1f} frame/s")

# Grafico a barre affiancate (scala logaritmica: le due misure possono differire di ordini di grandezza)
filtri = list(risultati.keys())
x = np.arange(len(filtri))
larghezza = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - larghezza / 2, [risultati[f][0] for f in filtri], larghezza, label="solo filtro", color="#4C72B0")
b2 = ax.bar(x + larghezza / 2, [risultati[f][1] for f in filtri], larghezza, label="catena completa", color="#C44E52")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(filtri)
ax.set_ylabel("Throughput (frame/s, scala log)")
ax.set_title("Throughput per filtro\n(i valori dipendono da hardware, codec e risoluzione del video)")
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.05,
                f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()


## 8. Esercizio — Frame rate, numero di frame e durata

La funzione `double_frame` scrive **due volte** ogni frame letto e imposta per l'output un frame rate **doppio** rispetto all'input.

Prima di eseguire le celle seguenti, prova a rispondere:

1. Quanti frame conterrà il video di output?
2. Quale sarà la sua durata, rispetto all'originale?
3. Il movimento apparirà più fluido? Perché?

Poi esegui le celle e verifica le risposte confrontando i metadati dei due file e riproducendo il video prodotto. Le soluzioni sono in fondo alla sezione, nascoste: clicca su ciascuna domanda per aprirla.

Su macOS la scrittura di molti frame in rapida successione può produrre in output numerose righe `OpenCV: AVF: waiting to write video data.`: sono messaggi informativi di AVFoundation e non indicano un errore.


In [ ]:
def double_frame(input_path, output_path, codec=CODEC, verbose=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    if os.path.exists(output_path):
        os.remove(output_path)

    fps    = cap.get(cv2.CAP_PROP_FPS) * 2
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*codec)
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Verificare il codec (avc1 su macOS, mp4v su Linux/Colab)."
        )

    t_start = time.perf_counter()
    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
        out.write(frame)
        n_frame += 1

    cap.release()
    out.release()

    elapsed    = time.perf_counter() - t_start
    throughput = n_frame / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"Frame scritti  : {n_frame} / {total}")
        print(f"Tempo totale   : {elapsed:.2f} s")
        print(f"Throughput     : {throughput:.1f} frame/s")
        print(f"Output salvato : {output_path}")

    return n_frame, elapsed

In [ ]:
double_frame(VIDEO_PATH, "output_double_frame.mp4")

# Verifica: metadati del video prodotto a confronto con l'originale
for path in (VIDEO_PATH, "output_double_frame.mp4"):
    cap = cv2.VideoCapture(path)
    fps_v = cap.get(cv2.CAP_PROP_FPS)
    n_v   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f"{path:<26} fps = {fps_v:6.2f}   frame = {n_v:5d}   durata = {n_v / fps_v:6.2f} s")


### Soluzioni

Prova a rispondere prima di aprire le soluzioni.

<details>
<summary><b>1. Quanti frame conterrà il video di output?</b></summary>

Il **doppio** dei frame letti dal video di input: ogni frame viene scritto due volte. Con $N$ frame in ingresso, l'output ne contiene $2N$. Il numero esatto si legge nell'output della cella precedente (riga `frame =` del file prodotto).

</details>

<details>
<summary><b>2. Quale sarà la sua durata, rispetto all'originale?</b></summary>

La **stessa** dell'originale. La durata è il rapporto tra numero di frame e frame rate, e il raddoppio di entrambi si annulla:

$$d_{out} = \frac{2N}{2 \cdot fps} = \frac{N}{fps} = d_{in}$$

</details>

<details>
<summary><b>3. Il movimento apparirà più fluido? Perché?</b></summary>

**No.** Ogni frame originale resta sullo schermo per lo stesso tempo di prima ($2 \times \frac{1}{2 \cdot fps} = \frac{1}{fps}$): i due frame identici consecutivi non aggiungono alcuna informazione temporale, quindi la sequenza di immagini percepita è identica a quella originale.

Per ottenere un movimento realmente più fluido occorrerebbe *interpolare* frame intermedi, ad esempio stimando il moto tra frame consecutivi: è lo stesso principio della predizione del movimento usata nella compressione video.

</details>


## 9. Riepilogo

In questa lezione abbiamo costruito una pipeline di elaborazione video frame-by-frame con OpenCV:

- **Lettura e ispezione**: `cv2.VideoCapture` per accedere ai metadati (fps, risoluzione, durata) e ai singoli frame.
- **Filtri frame-by-frame**: quattro trasformazioni — scala di grigi, sfocatura gaussiana, rilevamento bordi (Canny), sogliatura adattiva.
- **Scrittura**: `cv2.VideoWriter` per salvare il video processato con codec e frame rate configurabili.
- **Pipeline in cascata**: composizione di più filtri in sequenza tramite `apply_pipeline`.
- **Overlay**: aggiunta di informazioni testuali su ogni frame con `cv2.putText`.
- **Throughput**: misurazione empirica del costo di ciascun filtro, separando il tempo di elaborazione da quello di decodifica e codifica.
- **Frame rate e durata**: relazione tra numero di frame, fps e durata, verificata sui metadati del video prodotto.